# Rethinking How We Choose Statistical Tests — Solution

*Decision Academy Challenge*

This notebook contains the data simulation, complete analysis, and solution.

## How the Data Was Generated

The data was simulated with a known structure so we can compare our statistical conclusions to the ground truth:
- **Control group** (n=50): drawn from a normal distribution, mean=50, SD=10
- **Treatment group** (n=50): a *bimodal mixture* — 55% non-responders (mean=42, SD=8) and 45% responders (mean=78, SD=8)
- **True treatment mean**: 0.55 × 42 + 0.45 × 78 = 58.2, so the **true effect is +8.2 points**
- Site effects were added (Site_A: 0, Site_B: -3, Site_C: +2)

The treatment genuinely works, but not for everyone — some participants respond strongly, while others see little benefit. This heterogeneous response is realistic for many medical and psychological interventions.

The question is: which statistical approach recovers this true effect?

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

In [ ]:
# DATA SIMULATION (the same process that was used for our CSV data)
np.random.seed(394)

N = 50  # sample size per group

# Control: normal
ctrl_scores = np.random.normal(50, 10, N)

# Treatment: bimodal mixture
n_nonresp = int(N * 0.55)  # 27 non-responders
n_resp = N - n_nonresp      # 23 responders
trt_scores = np.concatenate([
    np.random.normal(42, 8, n_nonresp),
    np.random.normal(78, 8, n_resp)
])
trt_scores = trt_scores[np.random.permutation(N)]  # shuffle

print(f"True treatment mean: {0.55*42 + 0.45*78:.1f}")
print(f"True difference: {0.55*42 + 0.45*78 - 50:.1f}")

## Part 2: Load the Challenge Data

In [ ]:
df = pd.read_csv("challenge_data.csv")
print(f"Shape: {df.shape}")
df.head(10)

In [ ]:
ctrl = df[df["group"] == "control"]["cognitive_score"]
trt  = df[df["group"] == "treatment"]["cognitive_score"]

print("=== Descriptive Statistics ===")
print(df.groupby("group")["cognitive_score"].describe())
print(f"\nMean difference (treatment - control): {trt.mean() - ctrl.mean():.2f}")

## Part 3: Visualization

This is where the key insight emerges. The treatment group is **bimodal**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

axes[0].hist(ctrl, bins=15, color="#0056d2", edgecolor="white", alpha=0.8)
axes[0].axvline(ctrl.mean(), color="#ab615d", linewidth=2, linestyle="--", label=f"Mean = {ctrl.mean():.1f}")
axes[0].set_title("Control Group", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Cognitive Score")
axes[0].set_ylabel("Frequency")
axes[0].legend()

axes[1].hist(trt, bins=15, color="#6f8a6d", edgecolor="white", alpha=0.8)
axes[1].axvline(trt.mean(), color="#ab615d", linewidth=2, linestyle="--", label=f"Mean = {trt.mean():.1f}")
axes[1].axvline(trt.median(), color="#b08a3a", linewidth=2, linestyle=":", label=f"Median = {trt.median():.1f}")
axes[1].set_title("Treatment Group", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Cognitive Score")
axes[1].legend()

plt.suptitle("Distribution of Cognitive Scores by Group", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Overlaid density plot
fig, ax = plt.subplots(figsize=(10, 5))
ctrl.plot.kde(ax=ax, color="#0056d2", linewidth=2, label="Control")
trt.plot.kde(ax=ax, color="#6f8a6d", linewidth=2, label="Treatment")
ax.axvline(ctrl.mean(), color="#0056d2", linestyle="--", alpha=0.5)
ax.axvline(trt.mean(), color="#6f8a6d", linestyle="--", alpha=0.5)
ax.set_xlabel("Cognitive Score")
ax.set_ylabel("Density")
ax.set_title("Density Comparison: Treatment is Bimodal", fontsize=14, fontweight="bold")
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

## Part 4: Normality Tests

In [ ]:
sw_ctrl = stats.shapiro(ctrl)
sw_trt  = stats.shapiro(trt)

print("=== Shapiro-Wilk Test ===")
print(f"Control:   W = {sw_ctrl.statistic:.4f}, p = {sw_ctrl.pvalue:.4f}")
print(f"Treatment: W = {sw_trt.statistic:.4f}, p = {sw_trt.pvalue:.6f}")
print()
print("Both groups show departures from normality.")
print("The treatment group's departure is driven by its bimodal structure.")
print()
print("A common approach at this point is to reason:")
print("'The data is not normal, so I should use Mann-Whitney U.'")
print("Let's see what happens when we try all three tests...")

## Part 5: The Three Tests

In [ ]:
# Student's t-test (equal variances assumed)
t_student = stats.ttest_ind(trt, ctrl, equal_var=True)

# Welch's t-test (unequal variances)
t_welch = stats.ttest_ind(trt, ctrl, equal_var=False)

# Mann-Whitney U
mw = stats.mannwhitneyu(trt, ctrl, alternative="two-sided")

print("=" * 60)
print("TEST RESULTS")
print("=" * 60)
print(f"Student's t-test:  t = {t_student.statistic:.4f}, p = {t_student.pvalue:.4f}  {'*** SIGNIFICANT' if t_student.pvalue < 0.05 else 'not significant'}")
print(f"Welch's t-test:    t = {t_welch.statistic:.4f}, p = {t_welch.pvalue:.4f}  {'*** SIGNIFICANT' if t_welch.pvalue < 0.05 else 'not significant'}")
print(f"Mann-Whitney U:    U = {mw.statistic:.1f},  p = {mw.pvalue:.4f}  {'*** SIGNIFICANT' if mw.pvalue < 0.05 else 'NOT significant'}")

print("\nCOMPARING TO THE TRUE EFFECT (+8.2 points):")
print("Both t-tests detect the effect (p ~ 0.03).")
print(f"Mann-Whitney concludes there is NO effect (p = {mw.pvalue:.2f}).")

print("\nIf you relied on the 'not normal -> use Mann-Whitney' heuristic, you would have missed a real treatment effect.")

## Part 6: Why Do the Tests Disagree?

### What each test actually measures

| Test | Tests for |
|------|----------|
| T-tests (Student's, Welch's) | Difference in **population means** |
| Mann-Whitney U | **Stochastic dominance**: P(X > Y) ≠ 0.5 |

### Why Mann-Whitney misses the effect

- 55% of treatment participants (non-responders) score around 40–45, **below** the control mean of 52
- 45% of treatment participants (responders) score around 75–80, well **above**

The **mean** is pulled up by the responders (+6.3 point advantage). But for **stochastic dominance**, the non-responders frequently lose when paired with controls, weakening the P(Treatment > Control) signal.

### Why the t-test works despite non-normality

The Central Limit Theorem guarantees that the **sampling distribution of the mean** is approximately normal for n = 50, regardless of the shape of the underlying data. The t-test doesn't need the data to be normal — it needs the means to be approximately normally distributed, which they are.

In [ ]:
# Visual explanation: why MW fails
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: the distributions
bins = np.linspace(20, 100, 30)
axes[0].hist(ctrl, bins=bins, alpha=0.6, color="#0056d2", label="Control", edgecolor="white")
axes[0].hist(trt, bins=bins, alpha=0.6, color="#6f8a6d", label="Treatment", edgecolor="white")
axes[0].axvline(ctrl.mean(), color="#0056d2", linewidth=2, linestyle="--")
axes[0].axvline(trt.mean(), color="#6f8a6d", linewidth=2, linestyle="--")
axes[0].set_title("Overlapping Distributions", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Cognitive Score")
axes[0].legend()

# Right: bootstrap the mean difference
boot_diffs = []
np.random.seed(0)
for _ in range(10000):
    bc = np.random.choice(ctrl, size=len(ctrl), replace=True)
    bt = np.random.choice(trt, size=len(trt), replace=True)
    boot_diffs.append(bt.mean() - bc.mean())

boot_diffs = np.array(boot_diffs)
ci_low, ci_high = np.percentile(boot_diffs, [2.5, 97.5])

axes[1].hist(boot_diffs, bins=50, color="#b08a3a", edgecolor="white", alpha=0.8)
axes[1].axvline(0, color="#ab615d", linewidth=2, linestyle="-", label="No effect")
axes[1].axvline(np.mean(boot_diffs), color="#1f1f1c", linewidth=2, linestyle="--", label=f"Mean diff = {np.mean(boot_diffs):.1f}")
axes[1].axvspan(ci_low, ci_high, alpha=0.2, color="#b08a3a", label=f"95% CI: [{ci_low:.1f}, {ci_high:.1f}]")
axes[1].set_title("Bootstrap Distribution of Mean Difference", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Treatment - Control (mean)")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Bootstrap 95% CI for mean difference: [{ci_low:.2f}, {ci_high:.2f}]")
print(f"The CI does not contain 0 — confirming the treatment effect.")

## Part 7 (Level 2 (Advanced)): ANCOVA

In [ ]:
import statsmodels.formula.api as smf

# Model 1: group + baseline
model1 = smf.ols('cognitive_score ~ C(group, Treatment(reference="control")) + baseline_score', data=df).fit()
print("=== ANCOVA: group + baseline_score ===")
print(model1.summary().tables[1])
print(f"\nTreatment effect: {model1.params.iloc[1]:.2f}, p = {model1.pvalues.iloc[1]:.4f}")

In [ ]:
# Model 2: group + baseline + site
model2 = smf.ols('cognitive_score ~ C(group, Treatment(reference="control")) + baseline_score + C(site)', data=df).fit()
print("=== ANCOVA: group + baseline_score + site ===")
print(model2.summary().tables[1])
print(f"\nTreatment effect: {model2.params.iloc[1]:.2f}, p = {model2.pvalues.iloc[1]:.4f}")
print()
print("KEY INSIGHT: Adding site to the model increases the treatment effect")
print("from ~6.3 to ~8.0 and makes it more significant (p = 0.007 vs 0.033).")
print("Site was a source of unexplained variance masking part of the treatment effect.")

## Comparison to the True Effect

| Approach | Estimated effect | p-value | True effect = 8.2 |
|----------|----------------|---------|--------------------|
| Mann-Whitney U | (not a mean difference) | 0.398 | Misses it entirely |
| Welch's t-test | +6.3 | 0.033 | Detects, underestimates |
| ANCOVA (+ baseline) | +6.3 | 0.028 | Detects, underestimates |
| ANCOVA (+ baseline + site) | +8.0 | 0.007 | Closest to truth |

The ANCOVA with site effects recovers an estimate of +8.0 — remarkably close to the true +8.2 — and does so with the strongest statistical evidence (p = 0.007).

### Key Takeaways

1. **"Not normal" does not mean "use Mann-Whitney."** Match the test to your question.
2. **Mann-Whitney tests stochastic dominance, not means.** With different distribution shapes, it answers a different question.
3. **T-tests are robust to non-normality** at n = 50 (Central Limit Theorem).
4. **Visualize first.** The bimodality was obvious in histograms.
5. **Design beats test selection.** ANCOVA recovered the true effect almost exactly — far better than any simple test.